# Stage 3 — ESCO Occupation Classification (Batch API)
## Notebook 3.1 of 7 — Create Batch Input Files

**Role in the pipeline:**
This is the first notebook in Stage 3. It prepares JSONL input files for the OpenAI Batch API. Each file contains one API request per job posting, packaging the vacancy's extracted skill labels (or description fallback), job title, and vacancy ID into a function-calling prompt that asks the LLM to assign an ESCO occupation code.

**Position in Stage 3 sequence:**
1. ▶ **3.1 — Create batch input files** ← *you are here*
2. 3.2 — Submit batch jobs and monitor status *(run after Batch API completes)*
3. 3.3 — Extract classification results from batch output
4. 3.4 — Split missing and complete classifications
5. 3.5 — Create batch input files for missing records
6. 3.6 — Submit batch jobs for missing records *(run after Batch API completes)*
7. 3.7 — Extract results for missing records

**Prerequisites:**
- Stage 2 output pickles present in `../data/stage_02/processed/output/`
- `OPENAI_API_KEY` set in `../.env`
- Classification schema JSON at `STAGE3_CLASSIFY_SCHEME` path
- Classification prompt text at `STAGE3_CLASSIFY_PROMPT` path

**Output:** JSONL batch input files written to `../data/stage_03/intermediate/input/`

## 3.1.2. Load packages and configuration

Loads standard libraries, the shared `general` config module, and Stage 2/3 helper modules. `%autoreload 2` ensures that any edits to `.py` modules in `../code/` are picked up automatically without restarting the kernel.

In [3]:
import pandas as pd
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append("../code")
import general as g
g.clean_memory()

Imports the Stage 2 and Stage 3 processing modules and the OpenAI client. `random` is used to select the model name (allows easy A/B testing between model versions).

In [4]:
import stage2 as st2
import stage3 as st3
from openai import OpenAI

## 3.1.3. Load processing log

Loads or creates the Stage 3 process tracker. `get_stage_process_df()` syncs it with the Stage 2 tracker — any file with `extract_status == 'complete'` in Stage 2 that is not yet present in Stage 3 gets registered here with blank status columns.

The tracker has 10 status columns covering the full async Batch API lifecycle:
`input_batch_path/status`, `job_id/status`, `output_batch_path/status`, `result_path/status`.

The commented-out lines below show how to manually remove a corrupt or stuck row from the tracker if needed (use with caution).

In [5]:
process_df = st3.get_stage_process_df(g.Config.STAGE2_PROCESS_PATH, g.Config.STAGE3_PROCESS_PATH)
process_df

,input_file,extract_path,input_batch_path,input_batch_status,job_id,job_status,output_batch_path,output_batch_status,result_path,result_status
0,ua-2024-01-01,../data/stage_02/processed/output\ua-2024-01-0...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3.1.4. Create JSONL input files

Initialises the OpenAI client using the API key from the `.env` configuration.

In [6]:
client = OpenAI(api_key=g.Config.OPENAI_API_KEY)

Main loop: iterates over every row in the tracker. Rows already marked `input_batch_status == 'created'` are skipped (idempotent). For new rows:

1. Loads the Stage 2 output pickle via `get_extracted_skills_df()` — returns a slim DataFrame with `id`, `title`, `desc`, `skills` columns.
2. Reads the function-calling JSON schema (`STAGE3_CLASSIFY_SCHEME`) and the system prompt (`STAGE3_CLASSIFY_PROMPT`).
3. Writes a JSONL batch input file to `STAGE3_INPUT_PATH` via `write_batch_file()`. Each line is one API request for one vacancy.
4. Updates the tracker with the batch file path and sets `input_batch_status = 'created'`.

In [7]:
process_df = st3.get_stage_process_df(g.Config.STAGE2_PROCESS_PATH, g.Config.STAGE3_PROCESS_PATH)

for _,row in process_df.iterrows():
    if row["input_batch_status"] == "created":
        continue
    else:
        file_path = process_df.loc[_, "extract_path"]
        skills_df = st3.get_extracted_skills_df(file_path)
        class_schema = st3.read_classification_schema(g.Config.STAGE3_CLASSIFY_SCHEME)
        class_prompt = st3.read_classification_prompt(g.Config.STAGE3_CLASSIFY_PROMPT)

        batch_file_path = os.path.join(g.Config.STAGE3_INPUT_PATH, process_df.loc[_, "input_file"] + ".jsonl")
        batch_file_path = st3.write_batch_file(batch_file_path, skills_df, class_schema, class_prompt, "gpt-4o-mini")

        process_df.loc[_, "input_batch_path"] = batch_file_path
        process_df.loc[_, "input_batch_status"] = "created"
        process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)

✔ built batch input:  ../data/stage_03/raw/input\ua-2024-01-01.jsonl


---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.